In [14]:
# In this case the local root of the repo is our working directory
DIRECTORY = '../../'
font = 'arial'


from Library.Build_Model import *

# We declare this function here and not in the
# function-storing python file to modify it easily
# as it can change the printouts of the methods
def printout(V, Stats, model): 
    # printing Stats
    print("R2 = %.2f (+/- %.2f) Constraint = %.2f (+/- %.2f)" % \
          (Stats.train_objective[0], Stats.train_objective[1],
           Stats.train_loss[0], Stats.train_loss[1]))
    Vout = tf.convert_to_tensor(np.float32(model.Y))
    Loss_norm, dLoss = Loss_Vout(V, model.Pout, Vout)
    print('Loss Targets', np.mean(Loss_norm))
    Loss_norm, dLoss = Loss_SV(V, model.S)
    print('Loss SV', np.mean(Loss_norm))
    Vin = tf.convert_to_tensor(np.float32(model.X))
    Pin = tf.convert_to_tensor(np.float32(model.Pin))
    if Vin.shape[1] == model.S.shape[1]: # special case
        Vin  = tf.linalg.matmul(Vin, tf.transpose(Pin), b_is_sparse=True)
    Loss_norm, dLoss = Loss_Vin(V, model.Pin, Vin, model.mediumbound,model)
    print('Loss Vin bound', np.mean(Loss_norm))
    Loss_norm, dLoss = Loss_Vpos(V, model)
    print('Loss V positive', np.mean(Loss_norm))

## QP solver

## One Sample

In [18]:
import shutil
import numpy as np
# Run Mechanistic model (no training) QP (quadratic program) or LP (linear program)
# using E. coli core simulation training sets and EB (or UB) bounds

# What you can change
seed = 10
np.random.seed(seed=seed)  
replication = "no_replicates"
experiment = 'mediabotJLF1' # the experiment name
trainname = f'{experiment}_UB_reduced_1' # the training set file name
size = int(trainname.split('_')[-1]) # number of runs must be lower than the number of element in trainname
timestep =  int(1.5*1.0e4) # LP 1.0e4 QP 1.0e5
learn_rate = 1.0 # LP 0.3 QP 1.0
decay_rate = 0.59 # only in QP, UB 0.333 EB 0.9
# End of What you can change

# Create model and run GD for X and Y randomly drawn from trainingfile
trainingfile = DIRECTORY+'Dataset_model/'+trainname

# Copy the model without size
# remove the size part if present
new_train_name = "_".join(trainname.split('_')[:-1])
trainingfile = './Dataset_model/'+new_train_name
# copy the model and paste with new name
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".npz", trainingfile + ".npz")
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".xml", trainingfile + ".xml")

print(f"Training file used: {trainingfile}")

biomass_reaction_name = 'BIOMASS_Ec_iML1515_core_75p37M'
# Get the target index for biomass reaction
original_parameter = TrainingSet()
original_parameter.load(trainingfile)
target_index = [r.id for r in original_parameter.model.reactions].index(biomass_reaction_name)
size = original_parameter.Y.shape[0]

model = Neural_Model(trainingfile = trainingfile, 
              objective=[biomass_reaction_name], 
              model_type = 'MM_QP', 
              timestep = timestep, 
              learn_rate = learn_rate, 
              decay_rate = decay_rate)

# Select a random subset of the training set (of specified size)
ID = np.random.choice(model.X.shape[0], size, replace=False)
model.X, model.Y = model.X[ID,:], model.Y[ID,:]

# Prints a summary of the model before running
model.printout()

# Runs the appropriate method
if model.model_type is 'MM_QP':
    Ypred, Stats = MM_QP(model, verbose=True)

# Printing results
printout(Ypred, Stats, model)

Training file used: ./Dataset_model/mediabotJLF1_UB_reduced
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_1.xml: True
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_1.xml: True
training file: ./Dataset_model/mediabotJLF1_UB_reduced
model type: MM_QP
model scaler: 0.0
model input dim: 26
model output dim: 1
model medium bound: UB
timestep: 15000
training set size (66, 26) (66, 1)
QP-Loss 1 0.00047080935 2.6079848e-05
QP-Loss 10 0.00045086382 2.79169e-06
QP-Loss 100 0.0004506652 2.7761212e-06
QP-Loss 1000 0.0004487645 2.6733915e-06
QP-Loss 2000 0.0004467781 2.6222529e-06
QP-Loss 3000 0.00044484937 2.5926322e-06
QP-Loss 4000 0.00044295177 2.5713823e-06
QP-Loss 5000 0.0004410754 2.5541865e-06
QP-Loss 6000 0.0004392159 2.5392626e-06
QP-Loss 7000 0.00043737085 2.5258125e-06
QP-Loss 8000 0.00043553885 2.5133113e-06
QP-Loss 9000 0.00043371913 2.501467

## Ten Samples

In [20]:
import shutil
import numpy as np
# Run Mechanistic model (no training) QP (quadratic program) or LP (linear program)
# using E. coli core simulation training sets and EB (or UB) bounds

# What you can change
seed = 10
np.random.seed(seed=seed)  
replication = "no_replicates"
experiment = 'mediabotJLF1' # the experiment name
trainname = f'{experiment}_UB_reduced_10' # the training set file name
size = int(trainname.split('_')[-1]) # number of runs must be lower than the number of element in trainname
timestep =  int(1.5*1.0e4) # LP 1.0e4 QP 1.0e5
learn_rate = 1.0 # LP 0.3 QP 1.0
decay_rate = 0.59 # only in QP, UB 0.333 EB 0.9
# End of What you can change

# Create model and run GD for X and Y randomly drawn from trainingfile
trainingfile = DIRECTORY+'Dataset_model/'+trainname

# Copy the model without size
# remove the size part if present
new_train_name = "_".join(trainname.split('_')[:-1])
trainingfile = './Dataset_model/'+new_train_name
# copy the model and paste with new name
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".npz", trainingfile + ".npz")
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".xml", trainingfile + ".xml")

print(f"Training file used: {trainingfile}")

biomass_reaction_name = 'BIOMASS_Ec_iML1515_core_75p37M'
# Get the target index for biomass reaction
original_parameter = TrainingSet()
original_parameter.load(trainingfile)
target_index = [r.id for r in original_parameter.model.reactions].index(biomass_reaction_name)
size = original_parameter.Y.shape[0]

model = Neural_Model(trainingfile = trainingfile, 
              objective=[biomass_reaction_name], 
              model_type = 'MM_QP', 
              timestep = timestep, 
              learn_rate = learn_rate, 
              decay_rate = decay_rate)

# Select a random subset of the training set (of specified size)
ID = np.random.choice(model.X.shape[0], size, replace=False)
model.X, model.Y = model.X[ID,:], model.Y[ID,:]

# Prints a summary of the model before running
model.printout()

# Runs the appropriate method
if model.model_type is 'MM_QP':
    Ypred, Stats = MM_QP(model, verbose=True)

# Printing results
printout(Ypred, Stats, model)

Training file used: ./Dataset_model/mediabotJLF1_UB_reduced
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_10.xml: True
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_10.xml: True
training file: ./Dataset_model/mediabotJLF1_UB_reduced
model type: MM_QP
model scaler: 0.0
model input dim: 26
model output dim: 1
model medium bound: UB
timestep: 15000
training set size (660, 26) (660, 1)
QP-Loss 1 0.0004674259 2.4478568e-05
QP-Loss 10 0.0004509476 3.2515823e-06
QP-Loss 100 0.0004507375 3.2424887e-06
QP-Loss 1000 0.0004487104 3.1835377e-06
QP-Loss 2000 0.00044656856 3.152296e-06
QP-Loss 3000 0.00044448132 3.1314494e-06
QP-Loss 4000 0.00044242668 3.1143566e-06
QP-Loss 5000 0.00044039628 3.0989904e-06
QP-Loss 6000 0.00043838643 3.0845783e-06
QP-Loss 7000 0.0004363949 3.0707824e-06
QP-Loss 8000 0.00043442054 3.0573744e-06
QP-Loss 9000 0.0004324624 3.04

## 120 samples

In [21]:
import shutil
import numpy as np
# Run Mechanistic model (no training) QP (quadratic program) or LP (linear program)
# using E. coli core simulation training sets and EB (or UB) bounds

# What you can change
seed = 10
np.random.seed(seed=seed)  
replication = "no_replicates"
experiment = 'mediabotJLF1' # the experiment name
trainname = f'{experiment}_UB_reduced_120' # the training set file name
size = int(trainname.split('_')[-1]) # number of runs must be lower than the number of element in trainname
timestep =  int(1.5*1.0e4) # LP 1.0e4 QP 1.0e5
learn_rate = 1.0 # LP 0.3 QP 1.0
decay_rate = 0.59 # only in QP, UB 0.333 EB 0.9
# End of What you can change

# Create model and run GD for X and Y randomly drawn from trainingfile
trainingfile = DIRECTORY+'Dataset_model/'+trainname

# Copy the model without size
# remove the size part if present
new_train_name = "_".join(trainname.split('_')[:-1])
trainingfile = './Dataset_model/'+new_train_name
# copy the model and paste with new name
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".npz", trainingfile + ".npz")
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".xml", trainingfile + ".xml")

print(f"Training file used: {trainingfile}")

biomass_reaction_name = 'BIOMASS_Ec_iML1515_core_75p37M'
# Get the target index for biomass reaction
original_parameter = TrainingSet()
original_parameter.load(trainingfile)
target_index = [r.id for r in original_parameter.model.reactions].index(biomass_reaction_name)
size = original_parameter.Y.shape[0]

model = Neural_Model(trainingfile = trainingfile, 
              objective=[biomass_reaction_name], 
              model_type = 'MM_QP', 
              timestep = timestep, 
              learn_rate = learn_rate, 
              decay_rate = decay_rate)

# Select a random subset of the training set (of specified size)
ID = np.random.choice(model.X.shape[0], size, replace=False)
model.X, model.Y = model.X[ID,:], model.Y[ID,:]

# Prints a summary of the model before running
model.printout()

# Runs the appropriate method
if model.model_type is 'MM_QP':
    Ypred, Stats = MM_QP(model, verbose=True)

# Printing results
printout(Ypred, Stats, model)

Training file used: ./Dataset_model/mediabotJLF1_UB_reduced
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_120.xml: True
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\mediabotJLF1_UB_reduced_120.xml: True
training file: ./Dataset_model/mediabotJLF1_UB_reduced
model type: MM_QP
model scaler: 0.0
model input dim: 26
model output dim: 1
model medium bound: UB
timestep: 15000
training set size (7920, 26) (7920, 1)
QP-Loss 1 0.00046764564 2.5226107e-05
QP-Loss 10 0.00045097884 3.3065362e-06
QP-Loss 100 0.00045076828 3.2910018e-06
QP-Loss 1000 0.0004487386 3.1860939e-06
QP-Loss 2000 0.0004465955 3.1311984e-06
QP-Loss 3000 0.00044450763 3.0981837e-06
QP-Loss 4000 0.00044245244 3.0739423e-06
QP-Loss 5000 0.00044042149 3.0540518e-06
QP-Loss 6000 0.0004384111 3.0367194e-06
QP-Loss 7000 0.00043641904 3.0209742e-06
QP-Loss 8000 0.0004344441 3.006289e-06
QP-Loss 9000 0.0004324852